# Path2-Broad-category specialty classification using GloVe Embeddings

This section implements the GloVe embedding branch of our clinical text mining 
pipeline, applied to Path 2 (broad specialty categories, grouped into a smaller 
number of clinically related classes). The goal is to test whether reducing 
specialty granularity improves classification robustness compared to the 
fine-grained approach in Path 1, following the same GloVe-based methodology 
for direct comparability.

**Pipeline summary:**
1. Loaded and examined `super_clinical_mtsamples.csv`, confirming its 
   structure and category groupings before reusing Path 1's cleaning logic
2. Tokenized transcription text using the same custom clinical stopword list 
   as Path 1
3. Reused the pretrained GloVe embeddings (glove.6B.100d, Stanford NLP) 
   already loaded, building document vectors via mean-pooling and tracking 
   the OOV rate
4. Tuned Linear SVC and Logistic Regression via Grid Search with 5-fold 
   cross-validation (searching C ∈ {0.01, 0.1, 1, 10, 100})
5. Evaluated the best model on a held-out test set

**Key results:**
- Grid Search best CV macro-F1: [TBD]
- Final held-out test macro-F1: [TBD]
- Strongest / weakest classes: [TBD]

**Purpose of this comparison:** contrasting Path 1 (fine-grained, 9 classes) 
and Path 2 (broad categories) results lets us evaluate the trade-off between 
classification granularity and model robustness — a key question for 
real-world clinical deployment, where broader categories may be easier to 
predict reliably but less clinically useful, while fine-grained categories 
are more useful but harder to classify accurately.

In [2]:
import pandas as pd
import numpy as np
import re

In [3]:
path2_df = pd.read_csv('../data/super_clinical_mtsamples.csv')

In [4]:
path2_df.shape

(4012, 7)

In [5]:
path2_df.head()

,description,medical_specialty,sample_name,transcription,keywords,medical_specialty_clean,clinical_superclass
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller...",Allergy / Immunology,internal_and_systemic_medicine
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh...",Bariatrics,surgical_and_procedural
2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart...",Bariatrics,surgical_and_procedural
3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple...",Cardiovascular / Pulmonary,internal_and_systemic_medicine
4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo...",Cardiovascular / Pulmonary,internal_and_systemic_medicine


In [6]:
path2_df.columns

Index(['description', 'medical_specialty', 'sample_name', 'transcription',
       'keywords', 'medical_specialty_clean', 'clinical_superclass'],
      dtype='str')

In [7]:
#Specialty categories
path2_df['medical_specialty'].value_counts()

medical_specialty
Surgery                       1088
Cardiovascular / Pulmonary     371
Orthopedic                     355
Radiology                      273
General Medicine               259
Gastroenterology               224
Neurology                      223
Urology                        156
Obstetrics / Gynecology        155
ENT - Otolaryngology            96
Neurosurgery                    94
Hematology - Oncology           90
Ophthalmology                   83
Nephrology                      81
Pediatrics - Neonatal           70
Pain Management                 61
Psychiatry / Psychology         53
Podiatry                        47
Dermatology                     29
Dentistry                       27
Cosmetic / Plastic Surgery      27
Physical Medicine - Rehab       21
Sleep Medicine                  20
Endocrinology                   19
Bariatrics                      18
Chiropractic                    14
Rheumatology                    10
Diets and Nutritions            10
Sp

In [8]:
path2_df['transcription'].isna().sum()

np.int64(0)

In [9]:
path2_df['clinical_superclass'].value_counts()

clinical_superclass
surgical_and_procedural           2072
internal_and_systemic_medicine    1186
diagnostic_and_pathology           289
neurology_and_behavioural          276
rehab_and_applied_health           189
Name: count, dtype: int64

In [10]:
path2_df['clinical_superclass'].isna().sum()

np.int64(0)

Tokenization

In [11]:
with open('../data/clinical-stopwords.txt', 'r') as f:
    clinical_stopwords = set(line.strip().lower() for line in f if line.strip())

print(f"Loaded {len(clinical_stopwords)} clinical stopwords")

Loaded 808 clinical stopwords


In [12]:
def tokenize_and_clean(text, stopword_set):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = re.findall(r'\b[a-z]+\b', text)
    tokens = [t for t in tokens if t not in stopword_set and len(t) > 1]
    return tokens

#Loading the GloVe from the stanford glove vectors 

In [13]:
embeddings_index = {}
with open('../data/glove/glove.6B.100d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

print(f"Loaded {len(embeddings_index)} real GloVe vectors.")

Loaded 400000 real GloVe vectors.


In [14]:
path2_df = pd.read_csv('../data/super_clinical_mtsamples.csv')

In [15]:
print(path2_df.shape)
path2_df.head()

(4012, 7)


,description,medical_specialty,sample_name,transcription,keywords,medical_specialty_clean,clinical_superclass
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller...",Allergy / Immunology,internal_and_systemic_medicine
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh...",Bariatrics,surgical_and_procedural
2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart...",Bariatrics,surgical_and_procedural
3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple...",Cardiovascular / Pulmonary,internal_and_systemic_medicine
4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo...",Cardiovascular / Pulmonary,internal_and_systemic_medicine


Tokenize the transcription text using your clinical stopwords

In [16]:
path2_df['tokens'] = path2_df['transcription'].apply(lambda t: tokenize_and_clean(t, clinical_stopwords))

In [17]:
path2_df[['transcription', 'tokens']].head()

,transcription,tokens
0,"SUBJECTIVE:, This 23-year-old white female pr...","[subjective, year, white, female, presents, co..."
1,"PAST MEDICAL HISTORY:, He has difficulty climb...","[past, history, difficulty, climbing, stairs, ..."
2,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","[history, present, illness, seen, abc, today, ..."
3,"2-D M-MODE: , ,1. Left atrial enlargement wit...","[mode, left, atrial, enlargement, left, atrial..."
4,1. The left ventricular cavity size and wall ...,"[left, ventricular, cavity, size, wall, thickn..."


Build document vectors using GloVe (mean pooling) and check OOV

In [18]:
def document_vector(tokens, embeddings, dim=100):
    valid_vectors = [embeddings[word] for word in tokens if word in embeddings]
    if not valid_vectors:
        return np.zeros(dim)
    return np.mean(valid_vectors, axis=0)

In [19]:
def oov_rate(tokens, embeddings):
    if not tokens:
        return 0
    in_vocab = sum(1 for t in tokens if t in embeddings)
    return 1 - (in_vocab / len(tokens))

In [20]:
#applying the GloVe to the tokens
X_path2 = np.array([document_vector(tokens, embeddings_index) for tokens in path2_df['tokens']])
oov_rates_path2 = path2_df['tokens'].apply(lambda t: oov_rate(t, embeddings_index))

print(f"Vector matrix shape: {X_path2.shape}")
print(f"Mean OOV rate: {oov_rates_path2.mean():.3f}")

Vector matrix shape: (4012, 100)
Mean OOV rate: 0.047


In [21]:
y_path2 = path2_df['clinical_superclass']

Training/Testing datasets

In [22]:
from sklearn.model_selection import train_test_split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_path2, y_path2, test_size=0.3, stratify=y_path2, random_state=42
)

imports for hyperparameter tuning

In [23]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

Grid search for linear SVC

In [24]:
svc_param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

svc_grid_path2 = GridSearchCV(
    LinearSVC(class_weight='balanced', max_iter=5000, random_state=42),
    param_grid=svc_param_grid,
    scoring='f1_macro',
    cv=5
)
svc_grid_path2.fit(X_train2, y_train2)

print("Best Linear SVC params:", svc_grid_path2.best_params_)
print(f"Best Linear SVC CV macro-F1: {svc_grid_path2.best_score_:.3f}")

Best Linear SVC params: {'C': 0.1}
Best Linear SVC CV macro-F1: 0.585


Grid search for logistic regression

In [25]:
lr_param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

lr_grid_path2 = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    param_grid=lr_param_grid,
    scoring='f1_macro',
    cv=5
)
lr_grid_path2.fit(X_train2, y_train2)

print("Best Logistic Regression params:", lr_grid_path2.best_params_)
print(f"Best Logistic Regression CV macro-F1: {lr_grid_path2.best_score_:.3f}")

Best Logistic Regression params: {'C': 1}
Best Logistic Regression CV macro-F1: 0.559


In [26]:
#Training  the best model
final_model_path2 = LinearSVC(C=0.1, class_weight='balanced', max_iter=5000, random_state=42)
final_model_path2.fit(X_train2, y_train2)
final_preds_path2 = final_model_path2.predict(X_test2)

In [27]:
#Evaluation
from sklearn.metrics import classification_report, f1_score

print("Final test macro-F1:", f1_score(y_test2, final_preds_path2, average='macro'))
print(classification_report(y_test2, final_preds_path2))

Final test macro-F1: 0.5719388688555781
                                precision    recall  f1-score   support

      diagnostic_and_pathology       0.35      0.52      0.42        87
internal_and_systemic_medicine       0.67      0.63      0.65       356
     neurology_and_behavioural       0.51      0.69      0.59        83
      rehab_and_applied_health       0.38      0.50      0.43        56
       surgical_and_procedural       0.81      0.73      0.77       622

                      accuracy                           0.67      1204
                     macro avg       0.55      0.61      0.57      1204
                  weighted avg       0.70      0.67      0.68      1204



#XGBoost Classifier

In [30]:
from sklearn.preprocessing import LabelEncoder

le_path2 = LabelEncoder()
y_path2_encoded = le_path2.fit_transform(y_path2)

In [31]:
y_train2_encoded = le_path2.transform(y_train2)
y_test2_encoded = le_path2.transform(y_test2)

In [ ]:
#Hypertarameter tuning
xgb_param_grid = {
    'max_depth': [3, 5],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1]
}

In [33]:
#Grid search 
from xgboost import XGBClassifier

xgb_grid_path2 = GridSearchCV(
    XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', random_state=42, n_jobs=1),
    param_grid=xgb_param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)
xgb_grid_path2.fit(X_train2, y_train2_encoded)

print("Best XGBoost params (Path 2):", xgb_grid_path2.best_params_)
print(f"Best XGBoost CV macro-F1 (Path 2): {xgb_grid_path2.best_score_:.3f}")

Best XGBoost params (Path 2): {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best XGBoost CV macro-F1 (Path 2): 0.411


In [34]:
#Evaluation
best_xgb_path2 = xgb_grid_path2.best_estimator_
xgb_preds_path2 = best_xgb_path2.predict(X_test2)

print("XGBoost final test macro-F1 (Path 2):", f1_score(y_test2_encoded, xgb_preds_path2, average='macro'))
print(classification_report(y_test2_encoded, xgb_preds_path2, target_names=le_path2.classes_))

XGBoost final test macro-F1 (Path 2): 0.44332625495969874
                                precision    recall  f1-score   support

      diagnostic_and_pathology       0.24      0.20      0.22        87
internal_and_systemic_medicine       0.64      0.61      0.63       356
     neurology_and_behavioural       0.44      0.42      0.43        83
      rehab_and_applied_health       0.43      0.11      0.17        56
       surgical_and_procedural       0.73      0.82      0.77       622

                      accuracy                           0.65      1204
                     macro avg       0.50      0.43      0.44      1204
                  weighted avg       0.63      0.65      0.64      1204

